# LOKLOK randomized ABM triangulation (Colab T4 or CPU)

This notebook keeps the existing within-persona 2x2x2 ABM (Pipeline A) and adds
two randomized-assignment designs, then compares them instead of averaging them.

**Pipelines**

- **A: within-persona full factorial.** Every persona runs in all eight A/B/C cells.
  Read from the saved v8 artifacts. This is the primary thesis result.
- **B: agent-randomized ABM.** Each simulated agent is randomly assigned to one
  cell. Network dynamics run with per-agent switches. Many independent worlds.
- **C: cluster-randomized ABM.** Treatment is assigned at the network stratum
  level. Everyone inside a cluster shares the same cell, so interference is
  part of the design.
- **B-design:** assignment randomization over the precomputed potential-outcome
  matrix, which has no interference. Comparing B-sim with B-design isolates
  interference.

**Controls**

- Placebo sham factor D is randomized but absent from the equations.
- Negative-control outcome: pre-treatment `opinion_init` trait.

**Qualitative bridge and E1-E3**

The 193-segment three-coder human gold is mapped onto the five mechanism codes
C01-C05 and their ABM levers.

- **E1:** evidence-weighted weight priors reweighting the existing 256-point
  joint sweep. Runs here as an explicitly labelled exploratory sensitivity
  because the frozen gate is not met.
- **E2:** evidence-informed persona-trait shifts, re-run in the randomized ABM.
  Also exploratory for the same reason.
- **E3:** respondent-level theme features vs BGE-PCA predictive comparison.
  **Not run:** it requires the G0 pass for PO provenance and student_id linkage
  through `rides_matched.csv`, which are absent from the archive.

**Gate:** at least 4 of the 12 mechanism codes must be `usable` (8 or more
positive human labels and kappa >= .60). The current gold has 0 usable codes,
so E1-E3 stay gated; the script computes eligibility automatically and reports
the exploratory versions separately.

Everything here is model-conditional. It does not become a human causal claim.

**Questions this answers**

1. Does the within-persona contrast survive between-agent randomization?
2. Does cluster-level assignment change the effect (interference)?
3. Are the mechanism rankings (A/B/C and BxC) stable across designs?
4. Are placebo and negative controls zero?
5. Which qualitative mechanism codes support which ABM levers, at what tier?
6. What do E1 and E2 say as exploratory sensitivities, and why is E3 gated?


In [ ]:
# 1. Installs and environment. T4 is optional; the dynamics are NumPy.
!nvidia-smi 2>/dev/null || echo "CPU runtime"
!pip -q install networkx
import numpy, scipy, pandas
print('numpy', numpy.__version__, '| scipy', scipy.__version__, '| pandas', pandas.__version__)


In [ ]:
# 2. Settings. Edit only this cell.
N_PERSONAS   = 500
N_SESSIONS   = 18
BASE_SEED    = 20261105
B_ER_WORLDS  = 20
B_WS_WORLDS  = 10
B_BA_WORLDS  = 10
C_WORLDS     = 20
E2_WORLDS    = 10
B_DESIGN_DRAWS = 400
GATE_MIN_USABLE = 4


In [ ]:
# 3. Unpack the embedded triangulation bundle. No upload.
import base64, sys, zipfile
from pathlib import Path

BUNDLE_B64 = "..."
work = Path('/content/abm_triangulation_work'); work.mkdir(parents=True, exist_ok=True)
zpath = work / 'abm_triangulation_bundle.zip'
zpath.write_bytes(base64.b64decode(BUNDLE_B64))
with zipfile.ZipFile(zpath) as z:
    z.extractall(work)
sys.path.insert(0, str(work))
print('extracted:', sorted(p.name for p in work.iterdir() if p.is_file()))


In [ ]:
# 4. Load the saved v8 artifacts, the codebook, and the human-gold summary.
import json, yaml
import numpy as np
import pandas as pd
import abm_randomized as abm

work = Path('/content/abm_triangulation_work')
npz = np.load(work / 'figure_data.npz', allow_pickle=True)
pr = json.loads((work / 'pipeline_results.json').read_text(encoding='utf-8'))
cb = yaml.safe_load((work / 'codebook_v1.yaml').read_text(encoding='utf-8'))
CODE_NAMES = {cid: (v.get('name', cid) if isinstance(v, dict) else v)
              for cid, v in cb['codes'].items()}
qual = json.loads((work / 'qual_validation_summary.json').read_text(encoding='utf-8'))

theta_real = npz['theta_real']
groups = npz['groups']
engagement_tensor = npz['tensor'][:, -1, 5, :]      # terminal engagement, 500 x 8
pre_trait = npz['tensor'][:, 0, 0, 0]               # pre-treatment opinion
pb_mean = float(pr['measured_anchor']['pb_mean'])
calibration_shift = pb_mean - float(np.mean(npz['sr_eval'][:, 0]))

print('theta_real:', theta_real.shape, '| groups:', len(np.unique(groups)))
print('saved SReval G0/G1:', round(float(np.mean(npz['sr_eval'][:,0])),3),
      round(float(np.mean(npz['sr_eval'][:,7])),3))
print('calibration shift:', round(calibration_shift,4))
print('human gold:', qual['human_gold']['units'], 'units,',
      'usable codes in gold few-shot:', qual['qwen_gold_fewshot_vs_human']['usable_codes'])


In [ ]:
# 5. Pipeline A: within-persona full factorial from the saved run.
A = abm.pipeline_a(npz, engagement_tensor)
print(pd.DataFrame([{k: round(v['mean'],4) for k,v in A['agg'].items()
                     if k.startswith('sr_eval') or k.startswith('engagement')}]).T.to_string())


In [ ]:
# 6. Pipeline B: agent-randomized ABM (ER + WS + BA worlds).
B_ER = abm.run_design(theta_real, N_PERSONAS, N_SESSIONS, BASE_SEED, graph='ER',
                      mode='agent', n_worlds=B_ER_WORLDS, groups=groups,
                      calibration_shift=calibration_shift)
B_WS = abm.run_design(theta_real, N_PERSONAS, N_SESSIONS, BASE_SEED, graph='WS',
                      mode='agent', n_worlds=B_WS_WORLDS, groups=groups,
                      calibration_shift=calibration_shift)
B_BA = abm.run_design(theta_real, N_PERSONAS, N_SESSIONS, BASE_SEED, graph='BA',
                      mode='agent', n_worlds=B_BA_WORLDS, groups=groups,
                      calibration_shift=calibration_shift)
B_worlds = B_ER['worlds'] + B_WS['worlds'] + B_BA['worlds']
B_agg = abm.aggregate_worlds(B_worlds)
print('B worlds:', len(B_worlds))
print('B SReval ATE:', round(B_agg['sr_eval_ATE']['mean'],4),
      '[', round(B_agg['sr_eval_ATE']['lo'],4), ',', round(B_agg['sr_eval_ATE']['hi'],4), ']')
print('B engagement ATE:', round(B_agg['engagement_ATE']['mean'],4))


In [ ]:
# 7. Pipeline C: cluster-randomized ABM. Clusters share the same cell.
C = abm.run_design(theta_real, N_PERSONAS, N_SESSIONS, BASE_SEED, graph='ER',
                   mode='cluster', n_worlds=C_WORLDS, groups=groups,
                   calibration_shift=calibration_shift)
print('C worlds:', C['n_worlds'])
print('C SReval ATE:', round(C['agg']['sr_eval_ATE']['mean'],4),
      '[', round(C['agg']['sr_eval_ATE']['lo'],4), ',', round(C['agg']['sr_eval_ATE']['hi'],4), ']')
print('C engagement ATE:', round(C['agg']['engagement_ATE']['mean'],4))


In [ ]:
# 8. B-design: assignment randomization over potential outcomes (no interference).
B_design_agg, B_design_rows = abm.design_based_randomization(
    npz, engagement_tensor, pre_trait, groups=groups,
    n_draws=B_DESIGN_DRAWS, seed=BASE_SEED)
print('B-design draws:', B_DESIGN_DRAWS)
print('B-design SReval ATE:', round(B_design_agg['sr_eval_ATE']['mean'],4),
      '[', round(B_design_agg['sr_eval_ATE']['lo'],4), ',', round(B_design_agg['sr_eval_ATE']['hi'],4), ']')


In [ ]:
# 9. Placebo, negative controls, and mechanism direction checks from B worlds.
controls = {}
for key in ['placebo_D_eval', 'placebo_D_comp', 'negative_negA', 'negative_negB',
            'negative_negC', 'mechanism_dbi_B', 'mechanism_brs_A', 'mechanism_echo_BC']:
    vals = [r[key] for r in B_worlds if key in r and r[key] is not None]
    if vals:
        controls[key] = dict(mean=float(np.mean(vals)),
                             sd=float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0)
controls_df = pd.DataFrame(controls).T
print(controls_df.round(5).to_string())
controls_df.to_csv('/content/abm_out/placebo_negative_controls.csv')


In [ ]:
# 10. Compare the pipelines and run the EST proposition checks.
A['agg']['n_worlds'] = 1
B_agg['n_worlds'] = len(B_worlds)
B_design_agg['n_worlds'] = B_DESIGN_DRAWS
C['agg']['n_worlds'] = C['n_worlds']
comp = abm.design_comparison(A['agg'], B_agg, B_design_agg, C['agg'])
print('\nDesign comparison:')
print(comp[['design','n_worlds','sr_eval_ATE','sr_eval_B','sr_eval_BC','engagement_ATE']].round(4).to_string())
comp.to_csv('/content/abm_out/design_comparison.csv', index=False)

# interference diagnostic: B-sim minus B-design on the SReval ATE
if 'B_agent_randomized' in comp.design.values and 'B_design_based' in comp.design.values:
    bsim = float(comp[comp.design=='B_agent_randomized'].sr_eval_ATE.iloc[0])
    bdes = float(comp[comp.design=='B_design_based'].sr_eval_ATE.iloc[0])
    print('\nInterference diagnostic (B-sim - B-design ATE):', round(bsim - bdes, 4))

align = abm.qual_alignment(qual, CODE_NAMES,
                           work / 'human_vs_qwen_zero_shot.csv',
                           work / 'human_vs_qwen_gold_fewshot.csv')
gate = abm.e_gate(qual, work / 'human_vs_qwen_gold_fewshot.csv',
                  min_usable=GATE_MIN_USABLE)
est = abm.est_verification(comp, align, gate, pr['reliability'])
print('\nEST proposition checks:')
print(est.to_string(index=False))
est.to_csv('/content/abm_out/est_verification.csv', index=False)


In [ ]:
# 11. Qualitative bridge: C01-C05 alignment, E1/E2 exploratory, E3 status.
import os
os.makedirs('/content/abm_out', exist_ok=True)
print('Mechanism alignment:')
print(align.to_string(index=False))
align.to_csv('/content/abm_out/qual_abm_alignment.csv', index=False)

print('\nGate:', json.dumps(gate, indent=2))

e1 = abm.e1_exploratory(pr, qual)
print('\nE1 (exploratory) weighted vs uniform sweep quantiles:')
print(json.dumps(e1, indent=2))

# E2 exploratory trait shifts: C01/C04 -> critical_scrutiny, C03 -> echo_susceptibility
total = qual['human_gold']['units']; npos = qual['human_gold']['positive_counts']
s_cs = (npos.get('C01',0) + 0.5) / (total + 1.0)
s_ec = (npos.get('C03',0) + 0.5) / (total + 1.0)
d_cs = float(np.clip(0.5 * (s_cs - 0.5), -0.25, 0.25))
d_ec = float(np.clip(0.5 * (s_ec - 0.5), -0.25, 0.25))
e2_runs = {}
if abs(d_cs) > 1e-9:
    e2_runs['critical_scrutiny_shift'] = abm.run_design(
        theta_real, N_PERSONAS, N_SESSIONS, BASE_SEED, graph='ER', mode='agent',
        n_worlds=E2_WORLDS, groups=groups, calibration_shift=calibration_shift,
        draw_shift=(4, d_cs))
if abs(d_ec) > 1e-9:
    e2_runs['echo_susceptibility_shift'] = abm.run_design(
        theta_real, N_PERSONAS, N_SESSIONS, BASE_SEED, graph='ER', mode='agent',
        n_worlds=E2_WORLDS, groups=groups, calibration_shift=calibration_shift,
        draw_shift=(2, d_ec))
e2 = {}
base_ate = B_ER['agg']['sr_eval_ATE']['mean']
for name, run in e2_runs.items():
    e2[name] = dict(delta=round(run['agg']['sr_eval_ATE']['mean'] - base_ate, 4),
                    ate_mean=round(run['agg']['sr_eval_ATE']['mean'], 4),
                    engagement_delta=round(run['agg']['engagement_ATE']['mean']
                                           - B_ER['agg']['engagement_ATE']['mean'], 4),
                    status='exploratory_gate_not_met')
e2['status'] = 'exploratory_gate_not_met'
print('\nE2 (exploratory) trait shifts vs baseline B ATE:', json.dumps(e2, indent=2))

e3 = dict(status='not_run', reason='requires G0 pass for PO provenance and '
          'student_id linkage via rides_matched.csv; unavailable in archive')

e_status = dict(gate=gate, e1=e1, e2=e2, e3=e3)
with open('/content/abm_out/e1e3_status.json','w',encoding='utf-8') as f:
    json.dump(e_status, f, indent=2)
print('\nE1-E3 status written to /content/abm_out/e1e3_status.json')


In [ ]:
# 12. Package every table and the summary for download.
summary = dict(
    metadata=dict(seed=BASE_SEED, n_personas=N_PERSONAS, n_sessions=N_SESSIONS,
                  worlds_B=len(B_worlds), worlds_C=C['n_worlds'],
                  design_draws=B_DESIGN_DRAWS),
    design_comparison=comp.to_dict(orient='records'),
    est_verification=est.to_dict(orient='records'),
    controls=controls,
    gate=gate,
    e1=e1, e2=e2, e3=e3,
)
with open('/content/abm_out/triangulation_summary.json','w',encoding='utf-8') as f:
    json.dump(summary, f, indent=2, default=str)

import shutil
from google.colab import files
shutil.make_archive('/content/loklok_abm_triangulation_results', 'zip', '/content/abm_out')
try:
    files.download('/content/loklok_abm_triangulation_results.zip')
except Exception:
    print('Download /content/loklok_abm_triangulation_results.zip from the file browser.')


## How to read the results

- `design_comparison.csv`: ATE and factorial effects per design with 2.5/97.5 percentiles.
- `est_verification.csv`: EST-P1/P2/P3/P4 checks with model-conditional verdicts.
- `placebo_negative_controls.csv`: sham factor D and pre-treatment trait; both should be near zero.
- `qual_abm_alignment.csv`: C01-C05 human support vs ABM levers and evidence tier.
- `e1e3_status.json`: gate, exploratory E1/E2, and the E3 not-run reason.
- `triangulation_summary.json`: everything in one file.

**Verdict vocabulary**

- `supported_within_model`: the sign holds across the compared designs inside the simulation.
- `design_dependent`: the effect depends on how assignment is done; report the difference.
- `not_eligible`: the frozen qualitative gate is not met; exploratory results are labelled as such.

Do not describe any of this as a human treatment effect. Do not pool pipeline
estimates into one average. Agreement is triangulation inside the model;
disagreement is a finding about interference or assignment design.
